# RAGBench: Pipeline Walkthrough

This notebook demonstrates the RAG pipeline end-to-end on a handful of
real questions from the benchmark: chunking a corpus, retrieving
relevant passages with three different retrieval methods, and
generating an answer with a local instruction-tuned model.

For the full ablation study across 9 configs and statistical
significance testing, see `02_ablation_analysis.ipynb` and `report/REPORT.md`.

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'src'))
sys.path.insert(0, '../src')

from ragbench.data_prep import load_corpus, load_qa_pairs
from ragbench.retrieval import build_chunks, RetrievalIndex
from ragbench.generation import generate_answer
from ragbench.metrics_generation import compute_generation_metrics
from ragbench.metrics_retrieval import compute_retrieval_metrics

corpus = load_corpus()
qa_pairs = load_qa_pairs()
print(f'corpus docs: {len(corpus)}, qa pairs: {len(qa_pairs)}')

corpus docs: 2000, qa pairs: 150


## 1. Chunking
Splitting each document into ~200-token fixed windows with overlap.

In [2]:
chunks = build_chunks(corpus, strategy_and_size := ('fixed', 200, 40)[0], 200, 40)
print(f'{len(corpus)} documents -> {len(chunks)} chunks')
print('\nExample chunk:\n', chunks[0].text[:300])

2000 documents -> 2152 chunks

Example chunk:
 Instability troubled the early years of Kublai Khan's reign. Ogedei's grandson Kaidu refused to submit to Kublai and threatened the western frontier of Kublai's domain. The hostile but weakened Song dynasty remained an obstacle in the south. Kublai secured the northeast border in 1259 by installing 


## 2. Build the retrieval index (dense + BM25, both live in the same object)

In [3]:
index = RetrievalIndex(chunks)
index._ensure_dense()
index._ensure_bm25()
print('Index built.')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Index built.


## 3. Compare retrieval methods on the same question

In [4]:
qa = qa_pairs[3]
print('QUESTION:', qa['question'])
print('GOLD ANSWER:', qa['gold_answer'])
print('GOLD DOC:', qa['gold_doc_id'])

for method in ['dense', 'bm25', 'hybrid']:
    retrieved = index.search(qa['question'], method=method, top_k=3)
    doc_ids = [c.doc_id for c in retrieved]
    metrics = compute_retrieval_metrics(doc_ids, qa['gold_doc_id'])
    print(f"\n[{method}] retrieved docs: {doc_ids}")
    print(f"[{method}] metrics: {metrics}")

QUESTION: In what city's Marriott did the Panthers stay?
GOLD ANSWER: San Jose
GOLD DOC: doc_00023

[dense] retrieved docs: ['doc_00023', 'doc_00009', 'doc_00015']
[dense] metrics: {'hit': True, 'reciprocal_rank': 1.0, 'ndcg': 1.0, 'precision': 0.3333333333333333}

[bm25] retrieved docs: ['doc_00060', 'doc_00015', 'doc_00050']
[bm25] metrics: {'hit': False, 'reciprocal_rank': 0.0, 'ndcg': 0.0, 'precision': 0.0}

[hybrid] retrieved docs: ['doc_00015', 'doc_00023', 'doc_00060']
[hybrid] metrics: {'hit': True, 'reciprocal_rank': 0.5, 'ndcg': 0.6309297535714575, 'precision': 0.3333333333333333}


## 4. Generate an answer and score it

In [5]:
retrieved = index.search(qa['question'], method='dense', top_k=3)
context_texts = [c.text for c in retrieved]
answer = generate_answer(qa['question'], context_texts)

print('GENERATED ANSWER:', answer)
print('SCORES:', compute_generation_metrics(answer, qa['gold_answers_all'], context_texts))

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


GENERATED ANSWER: San Jose
SCORES: {'exact_match': True, 'f1': 1.0, 'is_abstention': False, 'groundedness': 1.0}


## 5. A few more examples, side by side
Including one where retrieval likely misses -- worth seeing how the
model behaves when given the *wrong* context (does it hallucinate
confidently, or hedge?).

In [6]:
import pandas as pd

rows = []
for qa in qa_pairs[:10]:
    retrieved = index.search(qa['question'], method='dense', top_k=3)
    context_texts = [c.text for c in retrieved]
    answer = generate_answer(qa['question'], context_texts)
    ret_metrics = compute_retrieval_metrics([c.doc_id for c in retrieved], qa['gold_doc_id'])
    gen_metrics = compute_generation_metrics(answer, qa['gold_answers_all'], context_texts)
    rows.append({
        'question': qa['question'][:60],
        'gold': qa['gold_answer'],
        'generated': answer,
        'retrieval_hit': ret_metrics['hit'],
        'em': gen_metrics['exact_match'],
        'f1': round(gen_metrics['f1'], 2),
        'grounded': round(gen_metrics['groundedness'], 2),
    })

pd.DataFrame(rows)

,question,gold,generated,retrieval_hit,em,f1,grounded
0,How does the secondary theory say most cpDNA i...,linear,linear,True,True,1.00,1.00
1,What is the name of the two-tentacled cydippid...,Lampea,Lampea,True,True,1.00,1.00
2,What drove increased rental prices in East New...,residents willing to pay higher market rate fo...,landlords found new residents willing to pay h...,True,False,0.86,1.00
3,In what city's Marriott did the Panthers stay?,San Jose,San Jose,True,True,1.00,1.00
4,What is usual form of oxygen bound compounds?,oxides,Dioxygen,False,False,0.00,1.00
5,What previous work did Lavoisier experiments d...,phlogiston theory,I don't know,False,False,0.00,0.33
6,Where did the residents of Antioch flee to?,The city's residents fled to the north,the north,True,True,1.00,1.00
7,When was 7 Lincoln Square completed?,June 1979,June 1979,True,True,1.00,1.00
8,Which Doctor Who show was the largest simulcas...,50th anniversary special,The Chaser's War on Everything,False,False,0.00,1.00
9,What is the estimated cost of the V&A branded ...,£76 million,£76 million,True,True,1.00,1.00
